# Foody.vn Restaurant & Review Crawler
## Professional Data Collection Pipeline

---

### Pipeline Phases

| Phase | Description | Endpoint |
|:-----:|-------------|----------|
| 1 | **Restaurant Discovery** | `GET /Place/HomeListPlace` |
| 2 | **Review Pagination** | `GET /Review/ResLoadMore` |
| 3 | **Review Detail Enrichment** | `GET /Review/GetReviewInfo` |
| 4 | **Data Cleaning & Normalization** | — |
| 5 | **Export to Google Drive** | — |

### Output Files
```
/content/drive/MyDrive/Crawl data/Foody/
  restaurants_{timestamp}.csv
  restaurants_{timestamp}.json
  reviews_{timestamp}.csv
  reviews_{timestamp}.json
  review_details_raw_{timestamp}.json
  checkpoints/
    restaurants_checkpoint.csv / .json
    reviews_checkpoint.csv / .json
    review_images_checkpoint.csv / .json
    crawl_progress.json
```

### Use Cases
- Sentiment Analysis on Vietnamese restaurant reviews
- Recommendation Systems training data
- Product/Service Quality Assessment
- Deep Learning for NLP tasks

## 1. Setup
Install and verify required packages.

In [ ]:
# Install required packages (only needed once per Colab session)
!pip install requests pandas beautifulsoup4 tqdm -q
print('All packages installed successfully.')

In [ ]:
import os
import re
import json
import html
import time
import pprint
import logging
import warnings
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

print(f'pandas   : {pd.__version__}')
print(f'requests : {requests.__version__}')
print('Imports successful.')


## 2. Mount Google Drive

All crawled data will be persisted to:
```
/content/drive/MyDrive/Crawl data/Foody/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

## 3. Configuration

Adjust these parameters before running the crawler.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `end_page` | 100 | How many pages of restaurants to discover |
| `max_restaurants` | None | Cap total restaurants (None = unlimited) |
| `max_reviews_per_restaurant` | None | Cap reviews per restaurant |
| `sleep_between_requests` | 0.8s | Polite delay between API calls |
| `sleep_between_restaurants` | 2.0s | Delay between restaurants |

In [ ]:
# ── Run Timestamp ───────────────────────────────────────────────────────────
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run timestamp: {RUN_TIMESTAMP}')

# ── Output Directory ────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/Crawl_data/Foody'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

# ── Production / Test Mode ──────────────────────────────────────────────────
# Set TEST_MODE=True for quick smoke tests (crawls TEST_RESTAURANT_LIMIT restaurants).
# Set TEST_MODE=False for full production crawl (crawls TARGET_RESTAURANTS restaurants).
TEST_MODE             = False  # ← production
TEST_RESTAURANT_LIMIT = 2      # restaurants to crawl when TEST_MODE=True
TARGET_RESTAURANTS    = 300    # restaurants to crawl when TEST_MODE=False
DOWNLOAD_IMAGES       = False  # True = download images to disk (future use)

# ── Checkpoint / Resume ─────────────────────────────────────────────────────
# Periodically overwrite fixed-name checkpoint files so a Colab disconnect
# never loses more than CHECKPOINT_EVERY_RESTAURANTS restaurants of progress.
CHECKPOINT_EVERY_RESTAURANTS = 10     # overwrite checkpoint files every N restaurants
RESUME_FROM_CHECKPOINT       = True   # True = continue from last checkpoint if found
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Phase 1: Restaurant Discovery ──────────────────────────────────────────
# 100 pages × 12 per page = 1,200 candidates — stops early once TARGET is reached.
DISCOVERY_CONFIG: Dict[str, Any] = {
    'start_page': 1,
    'end_page': 100,           # Increase if 300 restaurants not found in 100 pages
    'count_per_page': 12,
    'lat': 10.823099,          # Ho Chi Minh City center latitude
    'lon': 106.629664          # Ho Chi Minh City center longitude
}

# ── Crawler Behaviour ──────────────────────────────────────────────────────
CRAWLER_CONFIG: Dict[str, Any] = {
    'max_restaurants':            TEST_RESTAURANT_LIMIT if TEST_MODE else TARGET_RESTAURANTS,
    'max_reviews_per_restaurant': None,   # None = crawl all reviews per restaurant
    'sleep_between_requests':     0.8,    # seconds — polite delay between API calls
    'sleep_between_restaurants':  2.0,    # seconds — delay between restaurants
    'max_retries':                3,
    'request_timeout':            15,     # seconds
    'save_raw_json':              True,
    'checkpoint_interval':        10      # save checkpoint every N restaurants
}

# ── Base URLs ──────────────────────────────────────────────────────────────
BASE_URL          = 'https://www.foody.vn'
DISCOVERY_API     = f'{BASE_URL}/__get/Place/HomeListPlace'
LOAD_MORE_API     = f'{BASE_URL}/__get/Review/ResLoadMore'
REVIEW_DETAIL_API = f'{BASE_URL}/__get/Review/GetReviewInfo'

print()
print('Configuration loaded.')
print(f'  Test mode          : {TEST_MODE}')
print(f'  Target restaurants : {CRAWLER_CONFIG["max_restaurants"]}')
print(f'  Pages to discover  : {DISCOVERY_CONFIG["start_page"]} -> {DISCOVERY_CONFIG["end_page"]}')
print(f'  Max reviews/res    : {CRAWLER_CONFIG["max_reviews_per_restaurant"] or "Unlimited"}')
print(f'  Request sleep      : {CRAWLER_CONFIG["sleep_between_requests"]}s')
print(f'  Restaurant sleep   : {CRAWLER_CONFIG["sleep_between_restaurants"]}s')
print(f'  Download images    : {DOWNLOAD_IMAGES}')
print(f'  Checkpoint every   : {CRAWLER_CONFIG["checkpoint_interval"]} restaurants')
print(f'  Checkpoint (resume): every {CHECKPOINT_EVERY_RESTAURANTS} restaurants -> {CHECKPOINT_DIR}')
print(f'  Resume from chkpt  : {RESUME_FROM_CHECKPOINT}')


In [ ]:
# ── Safe Variable Defaults ───────────────────────────────────────────────────
# Initialise all pipeline variables so later cells never crash with NameError
# even when run out of order or when a phase produces no data.

RESTAURANTS:        List[Dict[str, Any]] = []
SELECTED_RESTAURANTS: List[Dict[str, Any]] = []
ALL_REVIEWS:        List[Dict[str, Any]] = []
ALL_REVIEW_IMAGES:  List[Dict[str, Any]] = []
ALL_RAW_DETAILS:    List[Dict[str, Any]] = []
DF_RESTAURANTS:     pd.DataFrame = pd.DataFrame()
DF_REVIEWS:         pd.DataFrame = pd.DataFrame()
DF_REVIEW_IMAGES:   pd.DataFrame = pd.DataFrame()

# Checkpoint / resume state — populated by the "Resume Check" cell below.
RESUME_DATA:        Optional[Dict[str, Any]] = None
RESUME_START_INDEX: int = 0

print('Safe defaults initialised.')
print('  RESTAURANTS, SELECTED_RESTAURANTS, ALL_REVIEWS, ALL_REVIEW_IMAGES, ALL_RAW_DETAILS -> []')
print('  DF_RESTAURANTS, DF_REVIEWS, DF_REVIEW_IMAGES -> empty DataFrame')
print('  RESUME_DATA -> None, RESUME_START_INDEX -> 0')


## 3a. Checkpoint / Resume System

Fault-tolerance layer for long-running crawls on Google Colab.

- Every `CHECKPOINT_EVERY_RESTAURANTS` restaurants, the crawler **overwrites**
  a fixed set of checkpoint files (no timestamped accumulation):
  `restaurants_checkpoint.csv/json`, `reviews_checkpoint.csv/json`,
  `review_images_checkpoint.csv/json`, `crawl_progress.json`
- On restart, if `RESUME_FROM_CHECKPOINT = True` and a checkpoint exists,
  the crawl **continues from the next restaurant** instead of starting over.
- The final timestamped exports (`restaurants_{timestamp}.csv`, etc.) are
  produced exactly as before — checkpoints are a backup mechanism only.

In [ ]:
# ── Checkpoint Save / Load Functions ─────────────────────────────────────────

def save_checkpoint(
    restaurants: List[Dict[str, Any]],
    reviews: List[Dict[str, Any]],
    images: List[Dict[str, Any]],
    checkpoint_dir: str,
    restaurants_processed: int,
    reviews_collected: int,
    images_collected: int,
    last_restaurant_id: Any,
) -> None:
    """Overwrite fixed-name checkpoint CSV/JSON files plus crawl_progress.json.

    Always overwrites the same filenames (no timestamps) so the checkpoint
    directory never accumulates hundreds of files across a long crawl.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)

    def _dump(records: List[Dict[str, Any]], name: str) -> None:
        if not records:
            return
        df = pd.DataFrame(records)
        df.to_csv(os.path.join(checkpoint_dir, f'{name}_checkpoint.csv'), index=False, encoding='utf-8-sig')
        with open(os.path.join(checkpoint_dir, f'{name}_checkpoint.json'), 'w', encoding='utf-8') as f:
            json.dump(records, f, ensure_ascii=False, indent=2, default=str)

    _dump(restaurants, 'restaurants')
    _dump(reviews, 'reviews')
    _dump(images, 'review_images')

    progress = {
        'restaurants_processed':   restaurants_processed,
        'reviews_collected':       reviews_collected,
        'review_images_collected': images_collected,
        'last_restaurant_id':      last_restaurant_id,
        'last_checkpoint_time':    datetime.now().isoformat(timespec='seconds'),
    }
    with open(os.path.join(checkpoint_dir, 'crawl_progress.json'), 'w', encoding='utf-8') as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)

    print('─' * 29)
    print('Checkpoint saved')
    print(f'Restaurants : {restaurants_processed}')
    print(f'Reviews     : {reviews_collected}')
    print(f'Images      : {images_collected}')
    print('Path:')
    print(checkpoint_dir)
    print('─' * 29)


def load_checkpoint(checkpoint_dir: str) -> Optional[Dict[str, Any]]:
    """Load checkpointed datasets + progress metadata. Returns None if no checkpoint exists."""
    progress_path = os.path.join(checkpoint_dir, 'crawl_progress.json')
    if not os.path.exists(progress_path):
        return None

    with open(progress_path, 'r', encoding='utf-8') as f:
        progress = json.load(f)

    def _load(name: str) -> List[Dict[str, Any]]:
        path = os.path.join(checkpoint_dir, f'{name}_checkpoint.json')
        if not os.path.exists(path):
            return []
        with open(path, 'r', encoding='utf-8') as fh:
            return json.load(fh)

    return {
        'restaurants': _load('restaurants'),
        'reviews':     _load('reviews'),
        'images':      _load('review_images'),
        'progress':    progress,
    }


print('save_checkpoint and load_checkpoint defined.')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')


In [ ]:
# ── Resume Check — load checkpoint (if any) before crawling ─────────────────
RESUME_DATA:        Optional[Dict[str, Any]] = None
RESUME_START_INDEX: int = 0

if RESUME_FROM_CHECKPOINT:
    RESUME_DATA = load_checkpoint(CHECKPOINT_DIR)

if RESUME_DATA is not None:
    _p = RESUME_DATA['progress']
    RESUME_START_INDEX = int(_p.get('restaurants_processed', 0) or 0)

    print('Checkpoint found.')
    print()
    print(f'  Restaurants processed : {_p.get("restaurants_processed", 0):,}')
    print(f'  Reviews collected     : {_p.get("reviews_collected", 0):,}')
    print(f'  Review images         : {_p.get("review_images_collected", 0):,}')
    print(f'  Last restaurant ID    : {_p.get("last_restaurant_id")}')
    print(f'  Last checkpoint time  : {_p.get("last_checkpoint_time")}')
    print()
    print(f'Resuming crawl from restaurant #{RESUME_START_INDEX + 1}')
else:
    reason = 'RESUME_FROM_CHECKPOINT is False' if not RESUME_FROM_CHECKPOINT else 'no crawl_progress.json found'
    print(f'No checkpoint to resume from ({reason}).')
    print(f'  Checkpoint directory: {CHECKPOINT_DIR}')
    print('Starting a fresh crawl from restaurant #1.')


## 4. Core Utilities

Session factory and resilient HTTP request helper with exponential-backoff retry.

In [ ]:
def create_session() -> requests.Session:
    """Create a requests.Session with browser-like headers for Foody.vn."""
    session = requests.Session()
    session.headers.update({
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/124.0.0.0 Safari/537.36'
        ),
        'Accept': 'application/json, text/javascript, */*; q=0.01',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept-Encoding': 'gzip, deflate, br',
        'Referer': 'https://www.foody.vn/',
        'Origin': 'https://www.foody.vn',
        'X-Requested-With': 'XMLHttpRequest',
        'Connection': 'keep-alive',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-origin',
    })
    return session


def warmup_session(session: requests.Session) -> bool:
    """Visit the main city page to acquire session cookies before API calls."""
    warmup_url = 'https://www.foody.vn/ho-chi-minh'
    logger.info(f'Warming up session: {warmup_url}')
    resp = make_request(session, warmup_url, timeout=15, sleep_between=1.5)
    if resp is not None:
        cookie_names = list(session.cookies.keys())
        logger.info(f'Warm-up OK — HTTP {resp.status_code}, cookies acquired: {cookie_names}')
        return True
    logger.warning('Warm-up request failed. Proceeding without session cookies.')
    return False


def make_request(
    session: requests.Session,
    url: str,
    params: Optional[Dict[str, Any]] = None,
    max_retries: int = 3,
    timeout: int = 15,
    sleep_between: float = 0.8
) -> Optional[requests.Response]:
    """HTTP GET with exponential-backoff retry. Returns None on total failure."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(url, params=params, timeout=timeout)
            resp.raise_for_status()
            return resp

        except requests.exceptions.HTTPError as exc:
            code = exc.response.status_code if exc.response is not None else 0
            logger.warning(f'[HTTP {code}] attempt {attempt}/{max_retries}: {url}')
            if code == 429:
                time.sleep(sleep_between * 15)
            elif code >= 500:
                time.sleep(sleep_between * 3 * attempt)
            else:
                break  # 4xx errors are not retried

        except requests.exceptions.ConnectionError:
            logger.warning(f'[ConnectionError] attempt {attempt}/{max_retries}: {url}')
            time.sleep(sleep_between * 2 * attempt)

        except requests.exceptions.Timeout:
            logger.warning(f'[Timeout] attempt {attempt}/{max_retries}: {url}')
            time.sleep(sleep_between * attempt)

        except requests.exceptions.RequestException as exc:
            logger.error(f'[RequestException] {exc}')
            break

        if attempt < max_retries:
            time.sleep(sleep_between * attempt)

    logger.error(f'All {max_retries} attempts failed: {url}')
    return None


# ── Initialise session ──────────────────────────────────────────────────────
SESSION = create_session()
test_resp = make_request(SESSION, 'https://www.foody.vn', timeout=10, sleep_between=1.0)
if test_resp is not None:
    print(f'Connection OK — HTTP {test_resp.status_code}')
    time.sleep(1)
    warmup_session(SESSION)
else:
    print('WARNING: Could not reach foody.vn. Check your internet connection.')


## 5. Phase 1 — Restaurant Discovery

Paginate through `HomeListPlace` API to collect restaurant metadata.

**Endpoint:**
```
GET https://www.foody.vn/__get/Place/HomeListPlace
    ?page=1&lat=10.823099&lon=106.629664&count=12&type=1
```

In [ ]:
# ── Debug: Test Discovery API — Page 1 ─────────────────────────────────────
# Run this cell BEFORE Phase 1 to diagnose any API response issues.
# It fires a single request and prints full diagnostic info without side-effects.

def debug_discovery_page(session: requests.Session, page: int = 1) -> None:
    """Fire one discovery API request and print structured diagnostics."""
    t_ms = int(time.time() * 1000)
    params = {
        't':     t_ms,
        'page':  page,
        'lat':   DISCOVERY_CONFIG['lat'],
        'lon':   DISCOVERY_CONFIG['lon'],
        'count': DISCOVERY_CONFIG['count_per_page'],
        'type':  1,
    }

    print(f'Endpoint     : {DISCOVERY_API}')
    print(f'Params       : {params}')
    print()

    resp = make_request(SESSION, DISCOVERY_API, params=params, max_retries=1, timeout=15)

    if resp is None:
        print('ERROR: Request returned None (all retries failed). Check connectivity.')
        return

    ct   = resp.headers.get('Content-Type', 'N/A')
    text = resp.text.strip()

    print(f'Status       : {resp.status_code}')
    print(f'Content-Type : {ct}')
    print(f'Response len : {len(resp.text):,} chars')
    print(f'Cookies      : {list(session.cookies.keys())}')
    print()
    print(f'Response preview:\n{"-" * 50}')
    print(text[:500])
    print(f'{"-" * 50}')
    print()

    # Diagnosis
    if not text:
        print('DIAGNOSIS: Response body is EMPTY.')
        print('  -> Foody may require auth or has blocked this IP/Colab range.')
        return

    if text.lower().startswith('<!doctype') or '<html' in text[:200].lower():
        print('DIAGNOSIS: Response is HTML — anti-bot / CAPTCHA page detected.')
        print('  -> Solutions:')
        print('     1. Copy cookies from a real browser session (F12 → Application → Cookies)')
        print('     2. Try a different network / VPN')
        print('     3. Add a longer warm-up delay before API calls')
        return

    try:
        data = resp.json()
        items = data.get('Items', [])
        print('DIAGNOSIS: JSON OK')
        print(f'  Keys       : {list(data.keys())}')
        print(f'  Items count: {len(items)}')
        if items:
            print(f'\n  First item:\n{pprint.pformat(items[0], indent=4)}')
        else:
            print('  WARNING: Items list is empty — endpoint structure may have changed.')
    except (json.JSONDecodeError, ValueError) as exc:
        print(f'DIAGNOSIS: JSON parse failed — {exc}')
        print('  -> Response body is not valid JSON.')


print('Running Discovery API debug for page 1...')
print('=' * 60)
debug_discovery_page(SESSION, page=1)


In [ ]:
def _is_json_response(resp: requests.Response) -> bool:
    """Return True only when the response looks like parseable JSON."""
    if resp.status_code != 200:
        return False
    ct = resp.headers.get('Content-Type', '')
    if 'json' not in ct and 'javascript' not in ct:
        return False
    if not resp.text.strip():
        return False
    return True


def _log_bad_response(resp: requests.Response, page: int) -> None:
    """Log detailed diagnostics and save raw response to disk for inspection."""
    ct      = resp.headers.get('Content-Type', 'N/A')
    preview = resp.text[:500] if resp.text else ''
    is_html  = '<html' in resp.text[:200].lower() or '<!doctype' in resp.text[:50].lower()
    is_empty = not resp.text.strip()

    logger.error(
        f'Page {page}: non-JSON response\n'
        f'  URL         : {resp.url}\n'
        f'  Status      : {resp.status_code}\n'
        f'  Content-Type: {ct}\n'
        f'  Length      : {len(resp.text):,} chars\n'
        f'  Preview     : {preview!r}'
    )

    if is_empty:
        logger.error('  CAUSE: Response body is empty.')
    elif is_html:
        logger.error('  CAUSE: Response is HTML — likely an anti-bot / CAPTCHA page.')

    # Persist raw response so the user can inspect it without re-running
    debug_path = os.path.join(OUTPUT_DIR, f'debug_response_page{page}_{RUN_TIMESTAMP}.txt')
    try:
        with open(debug_path, 'w', encoding='utf-8') as fh:
            fh.write(f'URL: {resp.url}\n')
            fh.write(f'Status: {resp.status_code}\n')
            fh.write(f'Content-Type: {ct}\n\n')
            fh.write(resp.text)
        logger.info(f'  Debug response saved -> {debug_path}')
    except OSError as save_err:
        logger.warning(f'  Could not save debug response: {save_err}')


def discover_restaurants(
    session: requests.Session,
    discovery_cfg: Dict[str, Any],
    crawler_cfg: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """Paginate HomeListPlace API and return list of restaurant dicts."""
    all_restaurants: List[Dict[str, Any]] = []
    max_restaurants          = crawler_cfg.get('max_restaurants')
    sleep_t                  = crawler_cfg['sleep_between_requests']
    max_consecutive_failures = 3
    consecutive_failures     = 0

    start = discovery_cfg['start_page']
    end   = discovery_cfg['end_page']
    count = discovery_cfg['count_per_page']
    lat   = discovery_cfg['lat']
    lon   = discovery_cfg['lon']

    logger.info(f'Starting discovery: pages {start} to {end}')

    for page in tqdm(range(start, end + 1), desc='Discovering restaurants'):
        params = {
            't':     int(time.time() * 1000),  # required anti-cache timestamp
            'page':  page,
            'lat':   lat,
            'lon':   lon,
            'count': count,
            'type':  1,
        }

        resp = make_request(
            session, DISCOVERY_API, params=params,
            max_retries=crawler_cfg['max_retries'],
            timeout=crawler_cfg['request_timeout'],
            sleep_between=sleep_t
        )

        if resp is None:
            logger.warning(f'Page {page}: request failed entirely, skipping')
            consecutive_failures += 1
            if consecutive_failures >= max_consecutive_failures:
                logger.error(
                    f'Stopping after {max_consecutive_failures} consecutive request failures. '
                    'Check connectivity or Foody endpoint availability.'
                )
                break
            continue

        # Pre-parse validation — avoids misleading JSONDecodeError on HTML responses
        if not _is_json_response(resp):
            _log_bad_response(resp, page)
            consecutive_failures += 1
            if consecutive_failures >= max_consecutive_failures:
                logger.error(
                    f'Stopping early: {max_consecutive_failures} consecutive non-JSON responses.\n'
                    'Possible causes:\n'
                    '  1. Foody has changed or removed this API endpoint\n'
                    '  2. Anti-bot protection triggered (add browser cookies)\n'
                    '  3. Network / firewall is blocking the request\n'
                    '  4. The API now requires authentication\n'
                    f'Inspect debug_response_*.txt files in: {OUTPUT_DIR}'
                )
                break
            continue

        consecutive_failures = 0  # reset counter on a valid response

        try:
            data = resp.json()
        except (json.JSONDecodeError, ValueError) as exc:
            logger.error(f'Page {page}: JSON parse error — {exc}')
            consecutive_failures += 1
            if consecutive_failures >= max_consecutive_failures:
                logger.error('Stopping after repeated JSON parse failures.')
                break
            continue

        items = data.get('Items', [])
        if not items:
            logger.info(f'Page {page}: empty Items list, stopping discovery')
            break

        crawl_ts = datetime.now().isoformat()
        for item in items:
            url_path = item.get('Url', '')
            restaurant: Dict[str, Any] = {
                'restaurant_id':       item.get('Id') or item.get('RestaurantId'),
                'restaurant_name':     item.get('Name', ''),
                'restaurant_url':      url_path,
                'restaurant_full_url': f'{BASE_URL}{url_path}',
                'address':             item.get('Address', ''),
                'avg_rating':          item.get('AvgRating'),
                'total_reviews':       item.get('TotalReviews', 0),
                'latitude':            item.get('Latitude'),
                'longitude':           item.get('Longitude'),
                'crawl_timestamp':     crawl_ts,
            }
            all_restaurants.append(restaurant)

        logger.info(f'Page {page}: +{len(items)} restaurants (total: {len(all_restaurants)})')

        if max_restaurants and len(all_restaurants) >= max_restaurants:
            all_restaurants = all_restaurants[:max_restaurants]
            logger.info(f'Reached max_restaurants cap: {max_restaurants}')
            break

        time.sleep(sleep_t)

    # Deduplicate by restaurant_id while preserving order
    seen: set = set()
    unique: List[Dict[str, Any]] = []
    for r in all_restaurants:
        rid = r['restaurant_id']
        if rid not in seen:
            seen.add(rid)
            unique.append(r)

    if not unique:
        logger.error(
            'Discovery returned 0 restaurants.\n'
            'Run the debug cell above for a detailed single-page diagnosis.\n'
            f'Check any debug_response_*.txt files in: {OUTPUT_DIR}'
        )

    logger.info(f'Discovery complete: {len(unique)} unique restaurants')
    return unique


print('discover_restaurants (with pre-parse validation and consecutive-failure guard) defined.')


In [ ]:
# ── Run Restaurant Discovery ───────────────────────────────────────────────
print('Starting Phase 1: Restaurant Discovery...')
print('=' * 60)

if (RESUME_FROM_CHECKPOINT and 'RESUME_DATA' in globals()
        and RESUME_DATA is not None and RESUME_DATA.get('restaurants')):
    RESTAURANTS = RESUME_DATA['restaurants']
    print(f'Resumed restaurant list from checkpoint: {len(RESTAURANTS):,} restaurants')
    print('  -> Skipping live discovery to keep the same restaurant order/IDs as the checkpointed run.')
else:
    RESTAURANTS = discover_restaurants(
        session=SESSION,
        discovery_cfg=DISCOVERY_CONFIG,
        crawler_cfg=CRAWLER_CONFIG
    )

print()
print(f'Total restaurants discovered: {len(RESTAURANTS)}')

if RESTAURANTS:
    SELECTED_RESTAURANTS = RESTAURANTS[:TEST_RESTAURANT_LIMIT] if TEST_MODE else RESTAURANTS
    print(f'Selected for crawl       : {len(SELECTED_RESTAURANTS)} (TEST_MODE={TEST_MODE})')
    print(f'Sample restaurant        : {RESTAURANTS[0]}')
else:
    SELECTED_RESTAURANTS = []
    print()
    print('WARNING: No restaurants were discovered.')
    print('  1. Re-run the debug cell above and inspect its DIAGNOSIS output.')
    print(f'  2. Check debug_response_*.txt files saved in: {OUTPUT_DIR}')
    print('  3. If the response is HTML, add real browser cookies to SESSION.')
    print()
    print('Phases 2-3 (review crawl) will be skipped — nothing to crawl.')


## 6. Restaurant Dataset Export

Save discovered restaurants to **CSV** and **JSON** before review crawling begins.

**Schema:**
```
restaurant_id | restaurant_name | restaurant_url | restaurant_full_url
address | avg_rating | total_reviews | latitude | longitude | crawl_timestamp
```

In [ ]:
def save_restaurants_dataset(
    restaurants: List[Dict[str, Any]],
    output_dir: str,
    timestamp: str,
    save_raw_json: bool = True
) -> pd.DataFrame:
    """Deduplicate and save restaurants to CSV and optionally JSON."""
    if not restaurants:
        logger.warning('No restaurants to save.')
        return pd.DataFrame()

    df = pd.DataFrame(restaurants)
    df.drop_duplicates(subset=['restaurant_id'], keep='first', inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Ensure correct column order
    ordered_cols = [
        'restaurant_id', 'restaurant_name', 'restaurant_url', 'restaurant_full_url',
        'address', 'avg_rating', 'total_reviews', 'latitude', 'longitude',
        'crawl_timestamp'
    ]
    df = df[[c for c in ordered_cols if c in df.columns]]

    csv_path  = os.path.join(output_dir, f'restaurants_{timestamp}.csv')
    json_path = os.path.join(output_dir, f'restaurants_{timestamp}.json')

    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    logger.info(f'Saved: {csv_path}')

    if save_raw_json:
        df.to_json(json_path, orient='records', force_ascii=False, indent=2)
        logger.info(f'Saved: {json_path}')

    return df

In [ ]:
if 'RESTAURANTS' in globals() and RESTAURANTS:
    DF_RESTAURANTS = save_restaurants_dataset(
        restaurants=RESTAURANTS,
        output_dir=OUTPUT_DIR,
        timestamp=RUN_TIMESTAMP,
        save_raw_json=CRAWLER_CONFIG['save_raw_json']
    )
    print(f'Restaurants DataFrame shape: {DF_RESTAURANTS.shape}')
    DF_RESTAURANTS.head()
else:
    print('WARNING: RESTAURANTS is not defined or empty.')
    print('  -> Run Phase 1 (Restaurant Discovery) first.')
    DF_RESTAURANTS = pd.DataFrame()


## 7. Phase 2 — Review Collection

For each restaurant:
1. Fetch the HTML page to extract `initDataReviews` (initial 5 review IDs + total count)
2. Paginate `ResLoadMore` API until all review IDs are collected

**Load More Endpoint:**
```
GET https://www.foody.vn/__get/Review/ResLoadMore
    ?t={ms_timestamp}&ResId={id}&LastId={last}&Count=5&Type=1&fromOwner=&isLatest=true&ExcludeIds=
```

In [ ]:
def fetch_restaurant_html(
    session: requests.Session,
    full_url: str,
    crawler_cfg: Dict[str, Any]
) -> Optional[str]:
    """Fetch restaurant HTML page. Returns text or None on failure."""
    resp = make_request(
        session, full_url,
        max_retries=crawler_cfg['max_retries'],
        timeout=crawler_cfg['request_timeout'],
        sleep_between=crawler_cfg['sleep_between_requests']
    )
    return resp.text if resp is not None else None


def parse_init_data_reviews(html_content: str) -> Optional[Dict[str, Any]]:
    """Extract and parse the `initDataReviews` JavaScript variable from page HTML."""
    pattern = r'var\s+initDataReviews\s*=\s*(\{.*?\});'
    match = re.search(pattern, html_content, re.DOTALL)
    if not match:
        logger.debug('initDataReviews not found in HTML')
        return None
    try:
        return json.loads(match.group(1))
    except (json.JSONDecodeError, ValueError) as exc:
        logger.error(f'initDataReviews JSON parse error: {exc}')
        return None


def extract_initial_review_ids(
    init_data: Dict[str, Any]
) -> Tuple[List[int], int, int]:
    """Return (review_id_list, last_id, total_count) from initDataReviews."""
    review_ids: List[int] = []
    for item in init_data.get('Items', []):
        rid = item.get('Id') or item.get('ReviewId')
        if rid:
            review_ids.append(int(rid))
    last_id: int = init_data.get('LastId', 0) or 0
    total:   int = init_data.get('Total', 0) or 0
    return review_ids, last_id, total


print('HTML fetch and parse functions defined.')

In [ ]:
def load_more_reviews(
    session: requests.Session,
    res_id: int,
    last_id: int,
    crawler_cfg: Dict[str, Any],
    count: int = 5
) -> Tuple[List[Dict[str, Any]], int]:
    """Call ResLoadMore API. Returns (raw_review_items, new_last_id).

    Each item is the full dict from the API (includes Pictures, TotalPictures, etc.)
    """
    t_ms = int(time.time() * 1000)
    params = {
        't':          t_ms,
        'ResId':      res_id,
        'LastId':     last_id,
        'Count':      count,
        'Type':       1,
        'fromOwner':  '',
        'isLatest':   'true',
        'ExcludeIds': ''
    }

    resp = make_request(
        session, LOAD_MORE_API, params=params,
        max_retries=crawler_cfg['max_retries'],
        timeout=crawler_cfg['request_timeout'],
        sleep_between=crawler_cfg['sleep_between_requests']
    )

    if resp is None:
        return [], last_id

    try:
        data = resp.json()
    except (json.JSONDecodeError, ValueError) as exc:
        logger.error(f'ResLoadMore JSON parse error: {exc}')
        return [], last_id

    # Return FULL items, not just IDs — Pictures live here
    new_items: List[Dict[str, Any]] = []
    for item in data.get('Items', []):
        rid = item.get('Id') or item.get('ReviewId')
        if rid:
            new_items.append(item)

    new_last_id: int = data.get('LastId') or last_id
    return new_items, int(new_last_id)


print('load_more_reviews defined (returns full items including Pictures).')


In [ ]:
def crawl_review_ids(
    session: requests.Session,
    restaurant: Dict[str, Any],
    crawler_cfg: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """Collect all raw review items for a restaurant via HTML + ResLoadMore pagination.

    Returns the full raw item dicts (not just IDs) so callers can access
    Pictures, TotalPictures, ResId, and other fields directly.
    """
    res_id      = restaurant['restaurant_id']
    full_url    = restaurant['restaurant_full_url']
    max_reviews = crawler_cfg.get('max_reviews_per_restaurant')
    sleep_t     = crawler_cfg['sleep_between_requests']

    # Step 1: fetch page HTML
    html_content = fetch_restaurant_html(session, full_url, crawler_cfg)
    if not html_content:
        logger.warning(f'[{res_id}] HTML fetch failed')
        return []

    # Step 2: parse initDataReviews — keep full items, not just IDs
    init_data = parse_init_data_reviews(html_content)
    if not init_data:
        logger.warning(f'[{res_id}] initDataReviews not found')
        return []

    raw_items: List[Dict[str, Any]] = []
    for item in init_data.get('Items', []):
        rid = item.get('Id') or item.get('ReviewId')
        if rid:
            raw_items.append(item)

    last_id: int = init_data.get('LastId', 0) or 0
    total:   int = init_data.get('Total', 0) or 0

    logger.info(f'  [{res_id}] initial={len(raw_items)}, total={total}, last_id={last_id}')

    if max_reviews and len(raw_items) >= max_reviews:
        return raw_items[:max_reviews]

    # Step 3: paginate ResLoadMore — each call returns full item dicts
    count_per_call = 5
    max_iters = max(0, (total // count_per_call) + 2)

    for _ in range(max_iters):
        if not last_id:
            break
        if max_reviews and len(raw_items) >= max_reviews:
            break

        new_items, new_last_id = load_more_reviews(
            session, res_id, last_id, crawler_cfg, count=count_per_call
        )

        if not new_items:
            break

        raw_items.extend(new_items)

        if new_last_id == last_id:
            break
        last_id = new_last_id
        time.sleep(sleep_t)

    # Deduplicate while preserving order — key on review ID
    seen_ids: set = set()
    unique_items: List[Dict[str, Any]] = []
    for item in raw_items:
        rid = item.get('Id') or item.get('ReviewId')
        if rid not in seen_ids:
            seen_ids.add(rid)
            unique_items.append(item)

    if max_reviews:
        unique_items = unique_items[:max_reviews]

    logger.info(f'  [{res_id}] collected {len(unique_items)} unique raw review items')
    return unique_items


print('crawl_review_ids defined (returns full raw items including Pictures).')


## 8. Phase 3 — Review Detail Collection

For each collected review ID, fetch full review details via `GetReviewInfo`.

**Endpoint:**
```
GET https://www.foody.vn/__get/Review/GetReviewInfo?reviewId={id}
```

**Response fields captured:**
`Id, RestaurantID, UserID, Title, Comment, Food, Services, Atmosphere, Position, Price, CreatedOn, UpdatedOn, PostedByDevice, ReviewType`

In [ ]:
def get_review_detail(
    session: requests.Session,
    review_id: int,
    crawler_cfg: Dict[str, Any]
) -> Optional[Dict[str, Any]]:
    """Fetch a single review's full detail from GetReviewInfo API."""
    resp = make_request(
        session, REVIEW_DETAIL_API,
        params={'reviewId': review_id},
        max_retries=crawler_cfg['max_retries'],
        timeout=crawler_cfg['request_timeout'],
        sleep_between=crawler_cfg['sleep_between_requests']
    )
    if resp is None:
        return None
    try:
        return resp.json()
    except (json.JSONDecodeError, ValueError) as exc:
        logger.error(f'Review {review_id} JSON parse error: {exc}')
        return None


print('get_review_detail defined.')

In [ ]:
def extract_review_images(
    review_item: Dict[str, Any],
    restaurant_id: Any,
    restaurant_name: str,
    crawl_ts: str
) -> List[Dict[str, Any]]:
    """Extract image records from a raw review item (ResLoadMore / initDataReviews).

    Uses 'ResId' (ResLoadMore key) with fallback to 'RestaurantID' (GetReviewInfo key).
    Returns [] when the item has no Pictures.
    """
    pictures  = review_item.get('Pictures') or []
    review_id = review_item.get('Id') or review_item.get('ReviewId')
    # ResLoadMore uses 'ResId'; GetReviewInfo uses 'RestaurantID'
    res_id = (
        review_item.get('ResId') or
        review_item.get('RestaurantID') or
        restaurant_id
    )

    image_rows: List[Dict[str, Any]] = []
    for pic in pictures:
        image_rows.append({
            'image_id':          pic.get('Id'),
            'review_id':         review_id,
            'restaurant_id':     res_id,
            'restaurant_name':   restaurant_name,
            'image_url':         pic.get('Url', ''),
            'image_description': pic.get('Description', ''),
            'width':             pic.get('Width'),
            'height':            pic.get('Height'),
            'bg_color':          pic.get('BgColor', ''),
            'total_likes':       pic.get('TotalLikes', 0),
            'photo_detail_url':  pic.get('PhotoDetailUrl', ''),
            'crawl_timestamp':   crawl_ts,
        })
    return image_rows


def crawl_restaurant_reviews(
    session: requests.Session,
    restaurant: Dict[str, Any],
    crawler_cfg: Dict[str, Any]
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Crawl all reviews for one restaurant.

    Returns:
        (review_rows, image_rows, raw_details)

    Images are extracted from raw ResLoadMore items (which contain Pictures).
    Review rows are enriched with GetReviewInfo data (user info, scores, etc.).
    """
    res_id   = restaurant['restaurant_id']
    res_name = restaurant['restaurant_name']
    res_url  = restaurant['restaurant_url']
    sleep_t  = crawler_cfg['sleep_between_requests']

    # raw_items = full dicts from initDataReviews + ResLoadMore (include Pictures)
    raw_items = crawl_review_ids(session, restaurant, crawler_cfg)
    if not raw_items:
        return [], [], []

    # Debug: how many raw items carry Pictures?
    n_with_pics = sum(1 for item in raw_items if item.get('Pictures'))
    print(f'Raw reviews with Pictures: {n_with_pics} / {len(raw_items)}')
    if raw_items and n_with_pics == 0:
        sample_keys = list(raw_items[0].keys())
        sample_tp   = [item.get('TotalPictures') for item in raw_items[:5]]
        logger.info(f'  [{res_id}] Sample raw item keys: {sample_keys}')
        logger.info(f'  [{res_id}] TotalPictures sample: {sample_tp}')

    review_rows: List[Dict[str, Any]] = []
    image_rows:  List[Dict[str, Any]] = []
    raw_details: List[Dict[str, Any]] = []
    crawl_ts = datetime.now().isoformat()
    label    = res_name[:28] + '..' if len(res_name) > 30 else res_name

    for raw_item in tqdm(raw_items, desc=f'  {label}', leave=False):
        review_id = raw_item.get('Id') or raw_item.get('ReviewId')
        if not review_id:
            continue

        # ── Extract images from raw item ───────────────────────────────────
        # Pictures live in ResLoadMore / initDataReviews items, NOT in GetReviewInfo.
        imgs = extract_review_images(raw_item, res_id, res_name, crawl_ts)
        image_rows.extend(imgs)

        # ── Enrich with GetReviewInfo ──────────────────────────────────────
        detail = get_review_detail(session, int(review_id), crawler_cfg)
        if detail is None:
            detail = raw_item  # fallback: use raw item if detail fetch fails

        raw_details.append(detail)

        owner = detail.get('Owner') or {}

        # Combine pictures from both sources (raw item is authoritative)
        combined_pics = raw_item.get('Pictures') or detail.get('Pictures') or []

        row: Dict[str, Any] = {
            'review_id':           review_id,
            'restaurant_id':       detail.get('RestaurantID') or raw_item.get('ResId') or res_id,
            'restaurant_name':     res_name,
            'user_id':             detail.get('UserID') or raw_item.get('UserID') or owner.get('Id'),
            'user_name':           owner.get('DisplayName', '') or raw_item.get('UserName', ''),
            'user_avatar_url':     owner.get('Avatar', ''),
            'title':               detail.get('Title', '') or raw_item.get('Title', ''),
            'comment':             (detail.get('Comment') or detail.get('Description') or
                                    raw_item.get('Description', '')),
            'avg_rating':          detail.get('AvgRating') or raw_item.get('AvgRating'),
            'food_score':          detail.get('Food') or raw_item.get('Food'),
            'service_score':       detail.get('Services') or raw_item.get('Services'),
            'atmosphere_score':    detail.get('Atmosphere') or raw_item.get('Atmosphere'),
            'position_score':      detail.get('Position') or raw_item.get('Position'),
            'price_score':         detail.get('Price') or raw_item.get('Price'),
            'review_type':         detail.get('ReviewType') or raw_item.get('ReviewType'),
            'review_type_name':    detail.get('ReviewTypeName', '') or raw_item.get('ReviewTypeName', ''),
            'created_date_raw':    detail.get('CreatedDate', '') or raw_item.get('CreatedDate', ''),
            'created_on':          detail.get('CreatedOn') or raw_item.get('CreatedOn'),
            'updated_on':          detail.get('UpdatedOn') or raw_item.get('UpdatedOn'),
            'device_name':         detail.get('PostedByDevice', '') or raw_item.get('PostedByDevice', ''),
            'device_type':         detail.get('DeviceType') or raw_item.get('DeviceType'),
            'total_views':         detail.get('TotalViews') or raw_item.get('TotalViews'),
            'total_like':          detail.get('TotalLike') or raw_item.get('TotalLikes'),
            'total_comment':       detail.get('TotalComment') or raw_item.get('TotalComment'),
            'has_images':          len(combined_pics) > 0,
            'image_count':         len(combined_pics),
            'review_url':          f'{BASE_URL}{res_url}#review-{review_id}' if res_url else '',
            'source_restaurant_url': res_url,
            'crawl_timestamp':     crawl_ts,
        }
        review_rows.append(row)
        time.sleep(sleep_t)

    print(f'Extracted review images  : {len(image_rows)}')
    if image_rows:
        print(f'Sample image record      : {image_rows[0]}')
    if raw_items:
        print(f'Sample raw review Pictures: {raw_items[0].get("Pictures")}')

    return review_rows, image_rows, raw_details


print('extract_review_images and crawl_restaurant_reviews defined.')
print('Images are now extracted from raw ResLoadMore items (which carry Pictures).')


In [ ]:
def crawl_multiple_restaurants(
    session: requests.Session,
    restaurants: List[Dict[str, Any]],
    crawler_cfg: Dict[str, Any],
    output_dir: str,
    timestamp: str,
    existing_reviews: Optional[List[Dict[str, Any]]] = None,
    existing_images: Optional[List[Dict[str, Any]]] = None,
    start_index: int = 0,
    checkpoint_dir: Optional[str] = None,
    checkpoint_every: Optional[int] = None,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Crawl reviews for restaurants[start_index:], with checkpointing and resume support.

    Returns:
        (all_reviews, all_review_images, all_raw_details)

    `existing_reviews` / `existing_images` seed the running totals when resuming
    from a checkpoint — newly crawled rows are appended to them.
    `start_index` is the absolute index (into the FULL `restaurants` list) of the
    first restaurant to crawl in this run; restaurants before it were already
    processed in a prior run and are skipped.

    Progress snapshots are printed every ~10% of total restaurants (min every 50).
    Checkpoints overwrite fixed-name files every `checkpoint_every` restaurants
    (falls back to `crawler_cfg['checkpoint_interval']` when not provided).
    """
    all_reviews:     List[Dict[str, Any]] = list(existing_reviews or [])
    all_images:      List[Dict[str, Any]] = list(existing_images or [])
    all_raw_details: List[Dict[str, Any]] = []
    checkpoint_n   = checkpoint_every if checkpoint_every else crawler_cfg.get('checkpoint_interval', 10)
    sleep_rest     = crawler_cfg['sleep_between_restaurants']
    total          = len(restaurants)
    remaining      = restaurants[start_index:]
    n_remaining    = len(remaining)
    # Print a progress snapshot ~10 times across the run (at least every 50)
    snapshot_every = max(50, total // 10) if total >= 10 else 1

    print(f'Restaurants to process  : {n_remaining:,} (of {total:,} total, starting at #{start_index + 1})')
    print(f'Checkpoint interval     : every {checkpoint_n} restaurants')
    print(f'Progress snapshot every : {snapshot_every} restaurants')
    print()

    last_restaurant_id: Any = None

    for offset, restaurant in enumerate(tqdm(remaining, desc='Crawling restaurants')):
        i         = start_index + offset   # absolute index into `restaurants`
        processed = i + 1                  # 1-based count of restaurants processed so far
        res_name  = restaurant['restaurant_name']
        res_id    = restaurant['restaurant_id']
        last_restaurant_id = res_id
        logger.info(f'[{processed}/{total}] {res_name} (ID: {res_id})')

        rows, imgs, raws = crawl_restaurant_reviews(session, restaurant, crawler_cfg)
        all_reviews.extend(rows)
        all_images.extend(imgs)
        all_raw_details.extend(raws)

        logger.info(
            f'  -> {len(rows)} reviews, {len(imgs)} images '
            f'(running total: {len(all_reviews):,} reviews / {len(all_images):,} images)'
        )

        # Periodic checkpoint — overwrite fixed-name backup files
        if checkpoint_dir and (processed % checkpoint_n == 0 or processed == total):
            save_checkpoint(
                restaurants=restaurants,
                reviews=all_reviews,
                images=all_images,
                checkpoint_dir=checkpoint_dir,
                restaurants_processed=processed,
                reviews_collected=len(all_reviews),
                images_collected=len(all_images),
                last_restaurant_id=last_restaurant_id,
            )

        # Progress snapshot — human-readable mid-run summary
        if processed % snapshot_every == 0 or processed == total:
            print(f'\n── Progress [{processed}/{total} restaurants processed] ──────────')
            print(f'  Restaurants processed   : {processed:,} / {total:,}')
            print(f'  Reviews collected       : {len(all_reviews):,}')
            print(f'  Review images collected : {len(all_images):,}')
            print()

        time.sleep(sleep_rest)

    logger.info(
        f'Crawl complete — '
        f'{total:,} restaurants, '
        f'{len(all_reviews):,} reviews, '
        f'{len(all_images):,} images, '
        f'{len(all_raw_details):,} raw details'
    )
    return all_reviews, all_images, all_raw_details


print('crawl_multiple_restaurants defined (supports checkpoint/resume).')


In [ ]:
# ── Run Review Crawl ────────────────────────────────────────────────────────
# In TEST_MODE, SELECTED_RESTAURANTS is already limited to TEST_RESTAURANT_LIMIT.
# In production, SELECTED_RESTAURANTS = all discovered restaurants (up to TARGET_RESTAURANTS).
# To limit reviews per restaurant (faster test): 
#   CRAWLER_CONFIG['max_reviews_per_restaurant'] = 50

print('Starting Phase 2 & 3: Review Collection + Detail Enrichment...')
print('=' * 60)

target_restaurants = (
    SELECTED_RESTAURANTS
    if 'SELECTED_RESTAURANTS' in globals() and SELECTED_RESTAURANTS
    else (RESTAURANTS if 'RESTAURANTS' in globals() and RESTAURANTS else [])
)

_n_discovered = len(RESTAURANTS) if 'RESTAURANTS' in globals() else 0
print(f'Restaurants discovered  : {_n_discovered:,}')
print(f'Restaurants to crawl    : {len(target_restaurants):,}')

# Resume state — populated by the "Resume Check" cell (defaults to a fresh start)
_resume_start   = RESUME_START_INDEX if ('RESUME_START_INDEX' in globals() and RESUME_START_INDEX) else 0
_resume_reviews = RESUME_DATA['reviews'] if ('RESUME_DATA' in globals() and RESUME_DATA is not None) else []
_resume_images  = RESUME_DATA['images']  if ('RESUME_DATA' in globals() and RESUME_DATA is not None) else []
if _resume_start:
    print(f'Resuming from           : restaurant #{_resume_start + 1} '
          f'({len(_resume_reviews):,} reviews / {len(_resume_images):,} images already collected)')
print()

if not target_restaurants:
    print('WARNING: No restaurants to crawl. Run Phase 1 (Restaurant Discovery) first.')
    ALL_REVIEWS       = []
    ALL_REVIEW_IMAGES = []
    ALL_RAW_DETAILS   = []
elif _resume_start >= len(target_restaurants):
    print(f'All {len(target_restaurants):,} restaurants were already processed in a previous run.')
    print('  -> Skipping crawl; reusing checkpointed reviews/images as the final dataset.')
    ALL_REVIEWS       = list(_resume_reviews)
    ALL_REVIEW_IMAGES = list(_resume_images)
    ALL_RAW_DETAILS   = []
else:
    ALL_REVIEWS, ALL_REVIEW_IMAGES, ALL_RAW_DETAILS = crawl_multiple_restaurants(
        session=SESSION,
        restaurants=target_restaurants,
        crawler_cfg=CRAWLER_CONFIG,
        output_dir=OUTPUT_DIR,
        timestamp=RUN_TIMESTAMP,
        existing_reviews=_resume_reviews,
        existing_images=_resume_images,
        start_index=_resume_start,
        checkpoint_dir=CHECKPOINT_DIR,
        checkpoint_every=CHECKPOINT_EVERY_RESTAURANTS,
    )

# ── Summary — always printed, regardless of which branch above executed ─────
print()
print('─' * 60)
print('CRAWL PHASE COMPLETE')
print(f'  Restaurants discovered  : {_n_discovered:,}')
print(f'  Restaurants processed   : {len(target_restaurants):,}')
print(f'  Reviews collected       : {len(ALL_REVIEWS):,}')
print(f'  Review images collected : {len(ALL_REVIEW_IMAGES):,}')
print(f'  Raw details saved       : {len(ALL_RAW_DETAILS):,}')
print('─' * 60)

# Validation sample
if ALL_REVIEW_IMAGES:
    print(f'\nSample image record: {ALL_REVIEW_IMAGES[0]}')
else:
    print('\nNOTE: ALL_REVIEW_IMAGES is empty.')
    print('  Possible causes:')
    print('    1. None of the crawled reviews had attached photos.')
    print('    2. "Pictures" key absent from ResLoadMore items — check debug above.')
    if ALL_RAW_DETAILS:
        print(f'    3. GetReviewInfo sample keys: {list(ALL_RAW_DETAILS[0].keys())}')

# Save raw review details JSON
if CRAWLER_CONFIG['save_raw_json'] and ALL_RAW_DETAILS:
    raw_path = os.path.join(OUTPUT_DIR, f'review_details_raw_{RUN_TIMESTAMP}.json')
    with open(raw_path, 'w', encoding='utf-8') as f:
        json.dump(ALL_RAW_DETAILS, f, ensure_ascii=False, indent=2, default=str)
    print(f'\nRaw details saved: {raw_path}')


## 9. Data Cleaning & Normalization

Apply the following transformations:

| Step | Operation |
|------|-----------|
| 1 | `html.unescape()` — decode HTML entities |
| 2 | Whitespace normalization (`\\s+` → single space) |
| 3 | Duplicate removal by `review_id` |
| 4 | Datetime conversion (`created_on`, `updated_on`) |
| 5 | Null handling — drop rows with no `review_id` |
| 6 | UTF-8 cleaning — strip non-UTF-8 characters |
| 7 | Column standardization — lowercase snake_case |

In [ ]:
def clean_text(text: Any) -> str:
    """Unescape HTML, normalize whitespace, strip non-UTF-8 bytes."""
    if not text or not isinstance(text, str):
        return ''
    text = html.unescape(text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.encode('utf-8', errors='ignore').decode('utf-8')
    return text


def clean_reviews_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Full cleaning pipeline for reviews DataFrame."""
    df = df.copy()

    # 1. Text cleaning
    text_cols = ['title', 'comment', 'restaurant_name', 'user_name', 'device_name',
                 'review_type_name', 'created_date_raw', 'user_avatar_url', 'review_url',
                 'source_restaurant_url']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

    # 2. Datetime conversion
    for col in ['created_on', 'updated_on']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

    # 3. Numeric scores
    score_cols = ['avg_rating', 'food_score', 'service_score', 'atmosphere_score',
                  'position_score', 'price_score']
    for col in score_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 4. Integer counters
    int_cols = ['total_views', 'total_like', 'total_comment', 'image_count', 'device_type']
    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    # 5. IDs as nullable int
    for col in ['review_id', 'user_id', 'restaurant_id', 'review_type']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    # 6. Drop rows with no review_id
    before = len(df)
    df.dropna(subset=['review_id'], inplace=True)

    # 7. Deduplicate
    df.drop_duplicates(subset=['review_id'], keep='first', inplace=True)
    after = len(df)
    logger.info(f'Cleaning: {before} -> {after} rows (removed {before - after})')

    # 8. Column standardization
    df.columns = [c.lower().replace(' ', '_') for c in df.columns]

    # Enforce column order (unknown columns are appended at end)
    ordered = [
        'review_id', 'restaurant_id', 'restaurant_name', 'source_restaurant_url',
        'user_id', 'user_name', 'user_avatar_url',
        'title', 'comment',
        'avg_rating', 'food_score', 'service_score', 'atmosphere_score',
        'position_score', 'price_score',
        'review_type', 'review_type_name',
        'created_date_raw', 'created_on', 'updated_on',
        'device_name', 'device_type',
        'total_views', 'total_like', 'total_comment',
        'has_images', 'image_count', 'review_url',
        'crawl_timestamp',
    ]
    present  = [c for c in ordered if c in df.columns]
    leftover = [c for c in df.columns if c not in ordered]
    df = df[present + leftover]
    df.reset_index(drop=True, inplace=True)
    return df


print('clean_text and clean_reviews_dataframe defined.')


## 10. Data Export

Clean reviews and save to CSV (`utf-8-sig`) and JSON (UTF-8).

In [ ]:
def save_reviews_dataset(
    reviews: List[Dict[str, Any]],
    output_dir: str,
    timestamp: str,
    save_raw_json: bool = True
) -> pd.DataFrame:
    """Clean, deduplicate, and save reviews to CSV + JSON."""
    if not reviews:
        logger.warning('No reviews to save.')
        return pd.DataFrame()

    df = pd.DataFrame(reviews)
    df = clean_reviews_dataframe(df)

    csv_path  = os.path.join(output_dir, f'reviews_{timestamp}.csv')
    json_path = os.path.join(output_dir, f'reviews_{timestamp}.json')

    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    logger.info(f'Saved: {csv_path}')

    if save_raw_json:
        df_export = df.copy()
        for col in df_export.select_dtypes(include=['datetimetz', 'datetime64']).columns:
            df_export[col] = df_export[col].astype(str)
        for col in df_export.select_dtypes(include=['Int64']).columns:
            df_export[col] = df_export[col].astype(object).where(df_export[col].notna(), None)
        df_export.to_json(json_path, orient='records', force_ascii=False, indent=2)
        logger.info(f'Saved: {json_path}')

    logger.info(f'Reviews dataset: {len(df)} rows x {len(df.columns)} columns')
    return df


print('save_reviews_dataset defined.')

In [ ]:
def save_review_images_dataset(
    images: List[Dict[str, Any]],
    output_dir: str,
    timestamp: str,
    save_raw_json: bool = True
) -> pd.DataFrame:
    """Deduplicate and save review_images to CSV + JSON.

    Dedup key: image_id (primary), fallback to (review_id, image_url).
    """
    if not images:
        logger.warning('No review images to save.')
        return pd.DataFrame()

    df = pd.DataFrame(images)

    # IDs as nullable int
    for col in ['image_id', 'review_id', 'restaurant_id']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    # Numeric fields
    for col in ['width', 'height', 'total_likes']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    # Deduplicate: prefer image_id; fall back to (review_id, image_url)
    before = len(df)
    if df['image_id'].notna().any():
        df.drop_duplicates(subset=['image_id'], keep='first', inplace=True)
    else:
        df.drop_duplicates(subset=['review_id', 'image_url'], keep='first', inplace=True)
    after = len(df)
    logger.info(f'Images dedup: {before} -> {after} rows (removed {before - after})')

    # Enforce column order
    ordered_cols = [
        'image_id', 'review_id', 'restaurant_id', 'restaurant_name',
        'image_url', 'image_description',
        'width', 'height', 'bg_color',
        'total_likes', 'photo_detail_url',
        'crawl_timestamp'
    ]
    df = df[[c for c in ordered_cols if c in df.columns]]
    df.reset_index(drop=True, inplace=True)

    csv_path  = os.path.join(output_dir, f'review_images_{timestamp}.csv')
    json_path = os.path.join(output_dir, f'review_images_{timestamp}.json')

    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    logger.info(f'Saved: {csv_path}')

    if save_raw_json:
        df_export = df.copy()
        for col in df_export.select_dtypes(include=['Int64']).columns:
            df_export[col] = df_export[col].astype(object).where(df_export[col].notna(), None)
        df_export.to_json(json_path, orient='records', force_ascii=False, indent=2)
        logger.info(f'Saved: {json_path}')

    logger.info(f'Review images dataset: {len(df)} rows x {len(df.columns)} columns')
    return df


print('save_review_images_dataset defined.')


In [ ]:
# ── Export: reviews.csv ──────────────────────────────────────────────────────
if 'ALL_REVIEWS' in globals() and ALL_REVIEWS:
    DF_REVIEWS = save_reviews_dataset(
        reviews=ALL_REVIEWS,
        output_dir=OUTPUT_DIR,
        timestamp=RUN_TIMESTAMP,
        save_raw_json=CRAWLER_CONFIG['save_raw_json']
    )
    print(f'Reviews DataFrame shape  : {DF_REVIEWS.shape}')
    print(f'Columns: {list(DF_REVIEWS.columns)}')
else:
    print('WARNING: ALL_REVIEWS is not defined or is empty.')
    print('  -> Run the "Phase 2 & 3" cell first.')
    DF_REVIEWS = pd.DataFrame()

print()

# ── Export: review_images.csv ────────────────────────────────────────────────
if 'ALL_REVIEW_IMAGES' in globals() and ALL_REVIEW_IMAGES:
    DF_REVIEW_IMAGES = save_review_images_dataset(
        images=ALL_REVIEW_IMAGES,
        output_dir=OUTPUT_DIR,
        timestamp=RUN_TIMESTAMP,
        save_raw_json=CRAWLER_CONFIG['save_raw_json']
    )
    print(f'Review images DataFrame shape: {DF_REVIEW_IMAGES.shape}')
else:
    print('WARNING: ALL_REVIEW_IMAGES is not defined or is empty.')
    print('  -> No images were found in the crawled reviews (normal if reviews have no photos).')
    DF_REVIEW_IMAGES = pd.DataFrame()


In [ ]:
# ── Dataset Validation ───────────────────────────────────────────────────────
print('Dataset shapes:')

_shapes = {
    'DF_RESTAURANTS':   DF_RESTAURANTS   if 'DF_RESTAURANTS'   in globals() else pd.DataFrame(),
    'DF_REVIEWS':       DF_REVIEWS       if 'DF_REVIEWS'       in globals() else pd.DataFrame(),
    'DF_REVIEW_IMAGES': DF_REVIEW_IMAGES if 'DF_REVIEW_IMAGES' in globals() else pd.DataFrame(),
}
for name, df in _shapes.items():
    shape_str = f'{df.shape[0]:,} rows x {df.shape[1]} cols' if not df.empty else 'empty'
    print(f'  {name:<20} : {shape_str}')

print()
if 'DF_REVIEW_IMAGES' in globals() and not DF_REVIEW_IMAGES.empty:
    print('DF_REVIEW_IMAGES preview:')
    DF_REVIEW_IMAGES.head()
else:
    print('DF_REVIEW_IMAGES is empty — no image rows to preview.')


## 11. Exploratory Statistics

Summary analytics on the crawled dataset.

In [ ]:
def generate_summary_statistics(
    df_restaurants: pd.DataFrame,
    df_reviews: pd.DataFrame,
    df_images: Optional[pd.DataFrame] = None
) -> None:
    """Print structured summary statistics to stdout."""
    divider = '=' * 65
    section = '-' * 65

    print(divider)
    print('  FOODY.VN CRAWL SUMMARY STATISTICS')
    print(divider)

    # Totals
    n_rest   = len(df_restaurants) if df_restaurants is not None and not df_restaurants.empty else 0
    n_rev    = len(df_reviews)     if df_reviews     is not None and not df_reviews.empty     else 0
    n_images = len(df_images)      if df_images      is not None and not df_images.empty      else 0

    print(f'\n  Total restaurants : {n_rest:,}')
    print(f'  Total reviews     : {n_rev:,}')
    print(f'  Total images      : {n_images:,}')
    if n_rest > 0:
        print(f'  Avg reviews / restaurant : {n_rev / n_rest:.1f}')
    if n_rev > 0:
        print(f'  Avg images  / review     : {n_images / n_rev:.1f}')

    # Top 10 restaurants by review count
    if df_reviews is not None and not df_reviews.empty:
        rev_name_col = 'restaurant_name' if 'restaurant_name' in df_reviews.columns else None
        if rev_name_col:
            print(f'\n{section}')
            print('  TOP 10 RESTAURANTS BY REVIEW COUNT')
            print(section)
            top10 = (
                df_reviews
                .groupby(rev_name_col)['review_id']
                .count()
                .sort_values(ascending=False)
                .head(10)
                .rename('review_count')
                .reset_index()
            )
            print(top10.to_string(index=False))

        # Score distributions
        score_cols = [c for c in
            ['avg_rating', 'food_score', 'service_score', 'atmosphere_score',
             'position_score', 'price_score']
            if c in df_reviews.columns]

        if score_cols:
            print(f'\n{section}')
            print('  SCORE DISTRIBUTIONS')
            print(section)
            print(df_reviews[score_cols].describe().round(2).to_string())

        # Reviews per restaurant distribution
        id_col = 'restaurant_id' if 'restaurant_id' in df_reviews.columns else None
        if id_col:
            print(f'\n{section}')
            print('  REVIEWS PER RESTAURANT DISTRIBUTION')
            print(section)
            dist = (
                df_reviews
                .groupby(id_col)['review_id']
                .count()
                .describe()
                .round(1)
            )
            print(dist.to_string())

    # Restaurant rating distribution
    if df_restaurants is not None and not df_restaurants.empty and 'avg_rating' in df_restaurants.columns:
        print(f'\n{section}')
        print('  RESTAURANT AVG RATING DISTRIBUTION')
        print(section)
        bins   = [0, 2, 4, 6, 8, 10]
        labels = ['0-2', '2-4', '4-6', '6-8', '8-10']
        df_tmp = df_restaurants.copy()
        df_tmp['rating_bin'] = pd.cut(
            df_tmp['avg_rating'], bins=bins, labels=labels, include_lowest=True
        )
        print(df_tmp['rating_bin'].value_counts().sort_index().to_string())

    # Review images summary
    if df_images is not None and not df_images.empty:
        print(f'\n{section}')
        print('  REVIEW IMAGE SUMMARY')
        print(section)
        print(f'  Total images : {n_images:,}')
        if 'review_id' in df_images.columns:
            rev_with_imgs = df_images['review_id'].nunique()
            print(f'  Reviews with images : {rev_with_imgs:,}')
            if n_rev > 0:
                print(f'  % reviews with images: {rev_with_imgs / n_rev * 100:.1f}%')
        if 'restaurant_name' in df_images.columns:
            print()
            print('  Top restaurants by image count:')
            top_img = (
                df_images
                .groupby('restaurant_name')['image_id']
                .count()
                .sort_values(ascending=False)
                .head(10)
                .rename('image_count')
                .reset_index()
            )
            print(top_img.to_string(index=False))

    print(f'\n{divider}')


print('generate_summary_statistics defined.')


In [ ]:
_df_rest   = DF_RESTAURANTS   if 'DF_RESTAURANTS'   in globals() and not DF_RESTAURANTS.empty   else pd.DataFrame()
_df_rev    = DF_REVIEWS       if 'DF_REVIEWS'       in globals() and not DF_REVIEWS.empty       else pd.DataFrame()
_df_images = DF_REVIEW_IMAGES if 'DF_REVIEW_IMAGES' in globals() and not DF_REVIEW_IMAGES.empty else None

if _df_rest.empty and _df_rev.empty:
    print('WARNING: No data available for statistics.')
    print('  -> Run Phase 1 (Discovery) and Phase 2-3 (Reviews) first, then re-run export.')
else:
    generate_summary_statistics(
        df_restaurants=_df_rest,
        df_reviews=_df_rev,
        df_images=_df_images
    )


In [ ]:
# ── Additional Analytics ─────────────────────────────────────────────────────

if 'DF_REVIEWS' in globals() and not DF_REVIEWS.empty:
    print('=== Review Type Breakdown ===')
    if 'review_type_name' in DF_REVIEWS.columns:
        print(DF_REVIEWS['review_type_name'].value_counts().to_string())
    elif 'review_type' in DF_REVIEWS.columns:
        print(DF_REVIEWS['review_type'].value_counts().to_string())
    print()

    print('=== Posted By Device Breakdown ===')
    if 'device_name' in DF_REVIEWS.columns:
        print(DF_REVIEWS['device_name'].value_counts().head(10).to_string())
    print()

    if 'created_on' in DF_REVIEWS.columns:
        print('=== Reviews by Year ===')
        _year = pd.to_datetime(DF_REVIEWS['created_on'], errors='coerce').dt.year
        print(_year.value_counts().sort_index().to_string())
        print()

    print('=== Missing Value Summary (reviews) ===')
    _missing     = DF_REVIEWS.isnull().sum()
    _missing_pct = (_missing / len(DF_REVIEWS) * 100).round(1)
    print(pd.DataFrame({'missing_count': _missing, 'missing_pct': _missing_pct}).to_string())
else:
    print('WARNING: DF_REVIEWS is not defined or is empty.')
    print('  -> Run the "Data Export" cell first to populate DF_REVIEWS.')

print()
print('─' * 60)
print()

# ── Image Analytics ──────────────────────────────────────────────────────────
if 'DF_REVIEW_IMAGES' in globals() and not DF_REVIEW_IMAGES.empty:
    df_img = DF_REVIEW_IMAGES

    print('=== Review Image Analytics ===')
    print(f'Total image records : {len(df_img):,}')

    if 'review_id' in df_img.columns:
        n_reviews_with_images = df_img['review_id'].nunique()
        print(f'Unique reviews with images : {n_reviews_with_images:,}')
        if 'DF_REVIEWS' in globals() and not DF_REVIEWS.empty:
            pct = n_reviews_with_images / len(DF_REVIEWS) * 100
            print(f'% of reviews with images   : {pct:.1f}%')

    if 'restaurant_name' in df_img.columns:
        print()
        print('=== Top Restaurants by Image Count ===')
        top_img = (
            df_img
            .groupby('restaurant_name')['image_id' if 'image_id' in df_img.columns else df_img.columns[0]]
            .count()
            .sort_values(ascending=False)
            .head(10)
            .rename('image_count')
            .reset_index()
        )
        print(top_img.to_string(index=False))

    if 'total_likes' in df_img.columns:
        print()
        print('=== Image Likes Summary ===')
        print(df_img['total_likes'].describe().round(2).to_string())

    print()
    print('=== Missing Value Summary (review_images) ===')
    _img_missing     = df_img.isnull().sum()
    _img_missing_pct = (_img_missing / len(df_img) * 100).round(1)
    print(pd.DataFrame({'missing_count': _img_missing, 'missing_pct': _img_missing_pct}).to_string())
else:
    print('INFO: DF_REVIEW_IMAGES is empty — no review images were collected.')
    print('  (Expected when crawled reviews contain no attached photos.)')


## 12. Summary

---

### Files Saved

| File | Description |
|------|-------------|
| `restaurants_{timestamp}.csv` | Restaurant metadata, UTF-8 BOM |
| `restaurants_{timestamp}.json` | Restaurant metadata, JSON |
| `reviews_{timestamp}.csv` | Cleaned reviews, UTF-8 BOM |
| `reviews_{timestamp}.json` | Cleaned reviews, JSON |
| `review_details_raw_{timestamp}.json` | Raw API responses |
| `checkpoints/restaurants_checkpoint.csv/.json` | Checkpoint backup — overwritten every `CHECKPOINT_EVERY_RESTAURANTS` restaurants |
| `checkpoints/reviews_checkpoint.csv/.json` | Checkpoint backup — overwritten every `CHECKPOINT_EVERY_RESTAURANTS` restaurants |
| `checkpoints/review_images_checkpoint.csv/.json` | Checkpoint backup — overwritten every `CHECKPOINT_EVERY_RESTAURANTS` restaurants |
| `checkpoints/crawl_progress.json` | Resume metadata (counts, last restaurant ID, last checkpoint time) |

### Schema Reference

**restaurants.csv**
```
restaurant_id | restaurant_name | restaurant_url | restaurant_full_url
address | avg_rating | total_reviews | latitude | longitude | crawl_timestamp
```

**reviews.csv**
```
source_restaurant_id | source_restaurant_name | source_restaurant_url
review_id | user_id | title | comment
food_score | service_score | atmosphere_score | position_score | price_score
created_on | updated_on | posted_by_device | review_type | crawl_timestamp
```

### Next Steps
- **Sentiment Analysis**: Apply PhoBERT or multilingual BERT to `comment` field
- **Recommendation Systems**: Use `food_score`, `service_score` etc. as features
- **Quality Assessment**: Train DL models on score distributions
- **EDA**: Visualize rating trends over time using `created_on`

In [ ]:
# ── Final Report ─────────────────────────────────────────────────────────────
print('CRAWL RUN COMPLETE')
print('=' * 55)
print(f'Run ID             : {RUN_TIMESTAMP}')
print(f'Output dir         : {OUTPUT_DIR}')
print(f'Test mode          : {TEST_MODE}')

_target = (
    TEST_RESTAURANT_LIMIT
    if ('TEST_MODE' in globals() and TEST_MODE)
    else (TARGET_RESTAURANTS if 'TARGET_RESTAURANTS' in globals() else '?')
)
print(f'Target restaurants : {_target}')

_n_restaurants = len(DF_RESTAURANTS)   if 'DF_RESTAURANTS'   in globals() and DF_RESTAURANTS   is not None else 0
_n_reviews     = len(DF_REVIEWS)       if 'DF_REVIEWS'       in globals() and DF_REVIEWS       is not None else 0
_n_images      = len(DF_REVIEW_IMAGES) if 'DF_REVIEW_IMAGES' in globals() and DF_REVIEW_IMAGES is not None else 0

print(f'Restaurants saved  : {_n_restaurants:,}')
print(f'Reviews saved      : {_n_reviews:,}')
print(f'Review images      : {_n_images:,}')

if _n_restaurants > 0:
    print(f'Avg reviews/rest   : {_n_reviews / _n_restaurants:.1f}')
if _n_reviews > 0:
    print(f'Avg images/review  : {_n_images / _n_reviews:.2f}')

print()

# List all files from this run
print('Saved files:')
try:
    found = False
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        if RUN_TIMESTAMP in fname:
            fpath   = os.path.join(OUTPUT_DIR, fname)
            size_kb = os.path.getsize(fpath) / 1024
            print(f'  {fname:<55}  ({size_kb:,.1f} KB)')
            found = True
    if not found:
        print(f'  (No files from this run yet — OUTPUT_DIR: {OUTPUT_DIR})')
except OSError as e:
    print(f'  Could not list output directory: {e}')
